# Tracing a Medical Assistant RAG System chat with LangSmith

In this notebook, we will explore how to trace the different calls that occur in a LangChain Agent using LangSmith. We will use a familiar agent, employed in the LangChain section of the course, where a RAG system with medical information was created.

So, not only will we observe the traces of the agent, but we will also examine the traces of the retriever. Additionally, we'll inspect the query sent to the vectorial database and the returned results.
__________

## Installing libraries & Loading Dataset

We will download the dataset from the Hugging Face datasets library. It's a dataset with information about diseases.

In [1]:
from dotenv import load_dotenv, find_dotenv
import os
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
LANGCHAIN_API_KEY = os.getenv("LANGCHAIN_API_KEY")

In [2]:
from datasets import load_dataset
data = load_dataset("keivalya/MedQuad-MedicalQnADataset", split='train')

In [3]:
data = data.to_pandas()
data.head(10)

,qtype,Question,Answer
0,susceptibility,Who is at risk for Lymphocytic Choriomeningiti...,LCMV infections can occur after exposure to fr...
1,symptoms,What are the symptoms of Lymphocytic Choriomen...,LCMV is most commonly recognized as causing ne...
2,susceptibility,Who is at risk for Lymphocytic Choriomeningiti...,Individuals of all ages who come into contact ...
3,exams and tests,How to diagnose Lymphocytic Choriomeningitis (...,"During the first phase of the disease, the mos..."
4,treatment,What are the treatments for Lymphocytic Chorio...,"Aseptic meningitis, encephalitis, or meningoen..."
5,prevention,How to prevent Lymphocytic Choriomeningitis (L...,LCMV infection can be prevented by avoiding co...
6,information,What is (are) Parasites - Cysticercosis ?,Cysticercosis is an infection caused by the la...
7,susceptibility,Who is at risk for Parasites - Cysticercosis? ?,Cysticercosis is an infection caused by the la...
8,exams and tests,How to diagnose Parasites - Cysticercosis ?,"If you think that you may have cysticercosis, ..."
9,treatment,What are the treatments for Parasites - Cystic...,Some people with cysticercosis do not need to ...


In [4]:
#uncoment this line if you want to limit the size of the data.
data = data[0:250]

As you can see, the medical information in the dataset is well-organized, and to someone like me, who is not an expert in the field, it appears to be quite valuable. This information could be a useful addition to any general medicine book to support primary care doctors.

Load the langchain libraries to load the document.

In [5]:
from langchain_community.document_loaders import DataFrameLoader
from langchain_chroma import Chroma

The Document is in the Answer column, and the others columns are Metadata.

In [6]:
df_loader = DataFrameLoader(data, page_content_column="Answer")


In [7]:
df_document = df_loader.load()
display(df_document[:2])

[Document(metadata={'qtype': 'susceptibility', 'Question': 'Who is at risk for Lymphocytic Choriomeningitis (LCM)? ?'}, page_content='LCMV infections can occur after exposure to fresh urine, droppings, saliva, or nesting materials from infected rodents.  Transmission may also occur when these materials are directly introduced into broken skin, the nose, the eyes, or the mouth, or presumably, via the bite of an infected rodent. Person-to-person transmission has not been reported, with the exception of vertical transmission from infected mother to fetus, and rarely, through organ transplantation.'),
 Document(metadata={'qtype': 'symptoms', 'Question': 'What are the symptoms of Lymphocytic Choriomeningitis (LCM) ?'}, page_content='LCMV is most commonly recognized as causing neurological disease, as its name implies, though infection without symptoms or mild febrile illnesses are more common clinical manifestations. \n                \nFor infected persons who do become ill, onset of sympt

We can chunk the documents. The size to which we want to split the document is a design decision. The larger it is, the larger the prompt will be, and the slower the Model's response process.

We also need to consider the maximum prompt size and ensure that the document does not exceed it.

In [8]:
from langchain_text_splitters import CharacterTextSplitter

In [9]:
text_splitter = CharacterTextSplitter(chunk_size=1250,
                                      separator="\n",
                                      chunk_overlap=100)
texts = text_splitter.split_documents(df_document)

Created a chunk of size 2198, which is longer than the specified 1250
Created a chunk of size 1402, which is longer than the specified 1250


These warnings we see are because it can't perform the partition of the required size. This is because it waits for a page break to divide the text and does so when possible.

In [10]:
first_doc = texts[1]
print(first_doc.page_content)

LCMV is most commonly recognized as causing neurological disease, as its name implies, though infection without symptoms or mild febrile illnesses are more common clinical manifestations. 
                
For infected persons who do become ill, onset of symptoms usually occurs 8-13 days after exposure to the virus as part of a biphasic febrile illness. This initial phase, which may last as long as a week, typically begins with any or all of the following symptoms: fever, malaise, lack of appetite, muscle aches, headache, nausea, and vomiting. Other symptoms appearing less frequently include sore throat, cough, joint pain, chest pain, testicular pain, and parotid (salivary gland) pain.


## Initialize the Embedding Model and Vector DB

We load the text-embedding-ada-002 model from OpenAI.

Obtain Your LangChain API Key from your Personal->Settings Area in LangSmith panel.

![My Image](https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/langsmith_API_KEY.jpg?raw=true)

In [11]:
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"]="https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"]="pr-2601"
# --> Then, add your api key to your .env file

In [12]:
from langchain_openai import OpenAIEmbeddings

model_name = 'text-embedding-ada-002'

embed = OpenAIEmbeddings(
    model=model_name,
    openai_api_key=OPENAI_API_KEY
)

If you used the full dataset, the execution of this cell may take 3 to 5 minutes. If you reduced the number of records before, it should take only a few seconds.

In [13]:
directory_cdb = 'chromadb/'
chroma_db = Chroma.from_documents(
    df_document, embed, persist_directory=directory_cdb
)

We are going to create three objects.

* The language model, which can be any of those from OpenAI, the most common being gpt-3.5.
* The memory, responsible for keeping the prompt with all the necessary history.
* The retrieval, used to obtain information stored in ChromaDB.

In [14]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.0
)

# Modern LCEL approach replaces the deprecated RetrievalQA chain
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_prompt = ChatPromptTemplate.from_template(
    """Answer the following question based only on the provided context:

Context: {context}

Question: {question}

Answer:"""
)

retriever = chroma_db.as_retriever()

qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | qa_prompt
    | llm
    | StrOutputParser()
)

# Wrap in a function that returns a dict (for compatibility with the Tool)
def qa_invoke(query):
    result = qa_chain.invoke(query)
    return {"query": query, "result": result}

We can try the isolated Retrieval to see if the information it returns is relevant.




In [15]:
qa_invoke("Who is at risk of neurological diseases?")

{'query': 'Who is at risk of neurological diseases?',
 'result': 'Individuals who come into contact with urine, feces, saliva, or blood of wild mice, owners of pet mice or hamsters from contaminated colonies, human fetuses vertically infected from an infected mother, and laboratory workers handling infected animals are at risk of neurological diseases.'}

### Open the Langsmith website

When observing the Retriever on Langsmith is possible to see the input and the documents returned by it:

![My Image](https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_Retriever_1.jpg?raw=true)

In the image below you can observe the call to the Model where the full prompt and the response are displayed.

![My Image](https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_Retriever_2.jpg?raw=true)

## Conclusions

LangSmith is an incredibly useful tool for tracing and storing all the information generated when making calls to LangChain.

The experiment has been a small success. The Vectorial database has been configured and filled with information from the dataset. A LangChain agent has been created, and it has been able to retrieve information from the database only when necessary. Don't forget that our ChatBot has memory.



And you have all the information in a project stored in LangSmith!!!!



## Migration Summary: LangChain 0.x → LangChain 1.0+

| Old (LangChain 0.x) | New (LangChain 1.0+) | Notes |
|---------------------|----------------------|-------|
| `from langchain.document_loaders import DataFrameLoader` | `from langchain_community.document_loaders import DataFrameLoader` | Moved to langchain-community package |
| `from langchain.vectorstores import Chroma` | `from langchain_chroma import Chroma` | Separate langchain-chroma package |
| `from langchain.text_splitter import CharacterTextSplitter` | `from langchain_text_splitters import CharacterTextSplitter` | Separate langchain-text-splitters package |
| `from langchain.chat_models import ChatOpenAI` | `from langchain_openai import ChatOpenAI` | Moved to langchain-openai package |
| `from langchain.agents import Tool` | `from langchain_core.tools import Tool` | Moved to langchain-core package |
| `RetrievalQA.from_chain_type(...)` | LCEL chain with `ChatPromptTemplate | llm | StrOutputParser` | RetrievalQA removed in 1.0+, use LCEL chains |
| `ConversationBufferWindowMemory` | `langgraph.checkpoint.memory.MemorySaver` | Memory handled by LangGraph checkpointer |
| `AgentExecutor` + `create_react_agent` | `langgraph.prebuilt.create_react_agent` | LangGraph provides a simpler, more powerful agent API |
| `agent.invoke({"input": "..."})` | `agent.invoke({"messages": [("user", "...")]}, config=config)` | LangGraph uses messages format with thread config |
| `agent.memory.clear()` | New `thread_id` in config | Each thread_id is a separate conversation |